# Pruning Benchmark Analysis

Use this notebook to explore pruning results once sweeps are complete.

In [ ]:
from pathlib import Path
import pandas as pd
import json
import numpy as np
import plotly.express as px

CSV_PATH = Path('results/neurosuite_pruning.csv')
METRICS_ROOT = Path('results')
assert CSV_PATH.exists(), f'Missing CSV: {CSV_PATH}'
df = pd.read_csv(CSV_PATH)
drop_cols = [c for c in ['config_json', 'config_yaml', 'metrics_json'] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)
df.head()


In [ ]:
group_fields = ['task', 'strategy', 'amount']
metrics = ['post_acc', 'delta_post_acc', 'post_loss', 'sparsity']
agg = {m: ['mean', 'std', 'count'] for m in metrics if m in df.columns}
summary = df.groupby(group_fields, dropna=False).agg(agg)
summary.columns = ['_'.join(col).strip('_') for col in summary.columns]
summary = summary.reset_index()
summary.head()


In [ ]:
baseline = df[df['strategy'] == 'noise_prune'][['task', 'amount', 'seed', 'post_acc']]
comparisons = df.merge(baseline, on=['task', 'amount', 'seed'], suffixes=('', '_noise'))
comparisons['delta_vs_noise'] = comparisons['post_acc'] - comparisons['post_acc_noise']
comparisons[['task', 'strategy', 'amount', 'delta_vs_noise']].head()


In [ ]:
fig = px.line(summary, x='amount', y='post_acc_mean', color='strategy', facet_col='task', markers=True, title='Accuracy vs Sparsity')
fig


In [ ]:
records = []
for metrics_path in METRICS_ROOT.rglob('metrics.json'):
    try:
        with metrics_path.open('r') as fh:
            data = json.load(fh)
    except json.JSONDecodeError:
        continue
    data['run_dir'] = str(metrics_path.parent)
    records.append(data)
metrics_df = pd.DataFrame(records)
metrics_df.head()
